# Centralized Training Baseline on Standard CIFAR-10

This notebook trains a single model on standard CIFAR-10 data (centralized approach) with Train/Validation/Test splits.

In [ ]:
# Setup for Google Colab
try:
    import google.colab
    IN_COLAB = True
    print("Running in Google Colab")
    
    from google.colab import drive
    drive.mount('/content/drive')
    
    import os
    os.chdir('/content/drive/MyDrive/EnsembleFederatedLearning')
    
    !pip install -q torch torchvision scikit-learn matplotlib seaborn
    
except ImportError:
    IN_COLAB = False
    print("Running locally")

## Import Libraries

In [ ]:
import sys
import json
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from torch.utils.data import DataLoader, Subset, Dataset, ConcatDataset
import random
import time

sys.path.append('..')
from training.utils import get_model, set_seed

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## Load Configuration

In [ ]:
# Load configuration from JSON
with open('config.json', 'r') as f:
    CONFIG = json.load(f)

# Set random seeds
SEED = CONFIG['seed']
set_seed(SEED)
CONFIG['seed'] = SEED

print("Configuration:")
for k, v in CONFIG.items():
    print(f"  {k}: {v}")

## Create Dataset

In [ ]:
# Standard transforms (No Augmentation)
transform_train = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

# Load CIFAR-10
print("Loading CIFAR-10 dataset...")
train_set_full = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)
test_dataset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)

# Define split indices (e.g. 45k train, 5k val)
total_train_size = len(train_set_full)
train_size = int(0.9 * total_train_size)
val_size = total_train_size - train_size

# Use fixed generator for reproducibility of split
generator = torch.Generator().manual_seed(CONFIG['seed'])
train_dataset, val_dataset = torch.utils.data.random_split(train_set_full, [train_size, val_size], generator=generator)

print(f"Train set size: {len(train_dataset)} (no augmentation)")
print(f"Validation set size: {len(val_dataset)} (no augmentation)")
print(f"Test set size: {len(test_dataset)}")

## Create DataLoaders

In [ ]:
# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'], shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=2)

print(f"Number of training batches: {len(train_loader)}")

## Initialize Model and Training Setup

In [ ]:
# Initialize model
model = get_model(
    model_name=CONFIG['model_name'],
    num_classes=10,
    pretrained=CONFIG['pretrained']
).to(device)

# Training setup
optimizer = optim.SGD(model.parameters(), lr=CONFIG['lr'], momentum=0.9)
criterion = nn.CrossEntropyLoss()

# Learning rate scheduler
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)

print(f"Model: {CONFIG['model_name']}")
print(f"Learning rate: {CONFIG['lr']}")
print(f"Batch size: {CONFIG['batch_size']}")

## Training Loop

In [ ]:
# Training configuration
num_epochs = 50
train_losses = []
train_accs = []
val_losses = []
val_accs = []

print(f"Starting centralized training for {num_epochs} epochs...\n")
start_time = time.time()

for epoch in range(1, num_epochs + 1):
    # Training phase
    model.train()
    epoch_loss = 0.0
    correct = 0
    total = 0
    
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    
    avg_train_loss = epoch_loss / len(train_loader)
    train_acc = correct / total
    train_losses.append(avg_train_loss)
    train_accs.append(train_acc)
    
    # Validation phase
    model.eval()
    val_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
    
    avg_val_loss = val_loss / len(val_loader)
    val_acc = correct / total
    val_losses.append(avg_val_loss)
    val_accs.append(val_acc)
    
    # Update learning rate
    scheduler.step()
    
    # Print progress
    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch}/{num_epochs} - "
              f"Train Loss: {avg_train_loss:.4f}, Train Acc: {train_acc:.4f}, "
              f"Val Loss: {avg_val_loss:.4f}, Val Acc: {val_acc:.4f}")

training_time = time.time() - start_time

# Final Test Evaluation
print("\nRunning Final Test Evaluation...")
model.eval()
test_loss = 0.0
correct = 0
total = 0
with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
        test_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

final_test_acc = correct / total

print(f"\n{'='*70}")
print(f"Training Complete!")
print(f"{'='*70}")
print(f"Total training time: {training_time:.2f}s ({training_time/60:.2f} min)")
print(f"Final Train Accuracy: {train_accs[-1]:.4f}")
print(f"Final Validation Accuracy: {val_accs[-1]:.4f}")
print(f"Final Test Accuracy: {final_test_acc:.4f}")

## Visualize Training Progress

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Plot 1: Loss curves
ax = axes[0]
epochs_range = range(1, num_epochs + 1)
ax.plot(epochs_range, train_losses, 'o-', label='Train Loss', linewidth=2)
ax.plot(epochs_range, val_losses, 's-', label='Validation Loss', linewidth=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('Training and Validation Loss Over Epochs')
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 2: Accuracy curves
ax = axes[1]
ax.plot(epochs_range, train_accs, 'o-', label='Train Accuracy', linewidth=2)
ax.plot(epochs_range, val_accs, 's-', label='Validation Accuracy', linewidth=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('Accuracy')
ax.set_title('Training and Validation Accuracy Over Epochs')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim([0, 1])

plt.tight_layout()
plt.savefig('centralized_training_curves.png', dpi=300, bbox_inches='tight')
plt.show()

print("Training curves saved as 'centralized_training_curves.png'")


## Confusion Matrix on Test Set

In [ ]:
from sklearn.metrics import confusion_matrix

# CIFAR-10 class names
class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer', 
               'dog', 'frog', 'horse', 'ship', 'truck']

# Collect all predictions and true labels
all_predictions = []
all_labels = []

model.eval()
with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        _, predicted = outputs.max(1)
        
        all_predictions.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# Compute confusion matrix
cm = confusion_matrix(all_labels, all_predictions)

# Plot confusion matrix
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Plot 1: Confusion matrix with counts
ax = axes[0]
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=class_names, yticklabels=class_names,
            cbar_kws={'label': 'Count'})
ax.set_xlabel('Predicted Label', fontsize=12)
ax.set_ylabel('True Label', fontsize=12)
ax.set_title('Confusion Matrix (Counts)', fontsize=14, fontweight='bold')
plt.setp(ax.get_xticklabels(), rotation=45, ha='right')
plt.setp(ax.get_yticklabels(), rotation=0)

# Plot 2: Normalized confusion matrix (percentages)
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
ax = axes[1]
sns.heatmap(cm_normalized, annot=True, fmt='.2f', cmap='Greens', ax=ax,
            xticklabels=class_names, yticklabels=class_names,
            vmin=0, vmax=1, cbar_kws={'label': 'Proportion'})
ax.set_xlabel('Predicted Label', fontsize=12)
ax.set_ylabel('True Label', fontsize=12)
ax.set_title('Confusion Matrix (Normalized)', fontsize=14, fontweight='bold')
plt.setp(ax.get_xticklabels(), rotation=45, ha='right')
plt.setp(ax.get_yticklabels(), rotation=0)

plt.tight_layout()
plt.savefig('centralized_confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

# Analyze per-class performance
print("\nPer-class Performance:")
print(f"{'Class':<15} {'Precision':<12} {'Recall':<12} {'F1-Score':<12} {'Support':<10}")
print("="*65)

for i, class_name in enumerate(class_names):
    # True Positives, False Positives, False Negatives
    tp = cm[i, i]
    fp = cm[:, i].sum() - tp
    fn = cm[i, :].sum() - tp
    support = cm[i, :].sum()
    
    # Calculate metrics
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    
    print(f"{class_name:<15} {precision:<12.4f} {recall:<12.4f} {f1:<12.4f} {support:<10}")

# Overall metrics
overall_accuracy = np.trace(cm) / cm.sum()
print(f"\n{'Overall Accuracy':<15} {overall_accuracy:.4f}")

# Find most confused pairs
print("\nMost Confused Class Pairs:")
confused_pairs = []
for i in range(len(class_names)):
    for j in range(len(class_names)):
        if i != j and cm[i, j] > 0:
            confused_pairs.append((class_names[i], class_names[j], cm[i, j]))

confused_pairs.sort(key=lambda x: x[2], reverse=True)
for true_class, pred_class, count in confused_pairs[:10]:
    print(f"  {true_class:<12} → {pred_class:<12}: {count:>4} times")

## Save Results

In [ ]:
# Save results to file
results = {
    'method': 'centralized',
    'config': CONFIG,
    'training_time': training_time,
    'num_epochs': num_epochs,
    'final_train_acc': float(train_accs[-1]),
    'final_val_acc': float(val_accs[-1]),
    'final_test_acc': float(final_test_acc),
    'train_losses': [float(x) for x in train_losses],
    'train_accs': [float(x) for x in train_accs],
    'val_losses': [float(x) for x in val_losses],
    'val_accs': [float(x) for x in val_accs],
}

with open('centralized_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print("Results saved to 'centralized_results.json'")

# Save model checkpoint
torch.save({
    'epoch': num_epochs,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'val_acc': val_accs[-1],
    'test_acc': final_test_acc,
}, 'centralized_model_checkpoint.pth')

print("Model checkpoint saved to 'centralized_model_checkpoint.pth'")

## Summary

**Centralized Training Results:**
- Trains a single model on all data combined (all rotations)
- Provides upper bound for performance (full data access)
- No communication overhead
- No privacy preservation

**Key Metrics:**
- Final test accuracy on standard (0°) test set
- Performance variance across different rotations
- Training time and convergence speed

**Use as Baseline:**
This centralized model serves as the performance ceiling for comparison with:
- Federated Averaging (FedAvg)
- Ensemble Federated Learning with Clustering